# Week 4 - Noise Sensitivity Analysis

## Gaussian Noise Injection

In [6]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [7]:
df = pd.read_csv("../data/cleaned_ai4i.csv")

print(df.shape)
df.head()

(10000, 15)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,Type_enc
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,2
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,1
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,1
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,1
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,1


In [3]:
target = "Machine failure"

drop_cols = [
    "UDI",
    "Product ID",
    "Type",
    target
]

X = df.drop(columns=drop_cols, errors="ignore")
y = df[target]

print(X.shape)
print(X.dtypes)

(10000, 11)
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
Type_enc                     int64
dtype: object


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nTraining class distribution:")
print(y_train.value_counts())
print("\nTest class distribution:")
print(y_test.value_counts())

Training set: (8000, 11)
Test set: (2000, 11)

Training class distribution:
Machine failure
0    7729
1     271
Name: count, dtype: int64

Test class distribution:
Machine failure
0    1932
1      68
Name: count, dtype: int64


In [9]:
def inject_noise(X_test, noise_level):

    X_noisy = X_test.copy()

    numeric_cols = X_noisy.select_dtypes(include=[np.number]).columns

    # Convert numeric columns to float
    X_noisy[numeric_cols] = X_noisy[numeric_cols].astype(float)

    noise = np.random.normal(
        loc=0,
        scale=noise_level,
        size=X_noisy[numeric_cols].shape
    )

    X_noisy.loc[:, numeric_cols] += noise

    return X_noisy

In [10]:
X_noise = inject_noise(X_test,0.05)

print(X_noise.head())

      Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
2997           300.438775               309.855292             1345.015769   
4871           303.792539               312.373641             1513.058785   
3858           302.492882               311.418219             1558.949817   
951            295.591795               306.321683             1509.038779   
6463           300.478090               310.003828             1358.043887   

      Torque [Nm]  Tool wear [min]       TWF       HDF       PWF       OSF  \
2997    62.651029       153.042941  0.078106  0.015775  0.102924  0.042341   
4871    40.126329       135.027651 -0.004178 -0.027689  0.061874  0.083592   
3858    37.553975       208.985901 -0.007904  0.104668  0.005857 -0.025344   
951     35.792129        60.019685 -0.002692 -0.052793 -0.042002  0.007561   
6463    60.332760       102.024204  0.061908  0.000424 -0.034622  0.057964   

           RNF  Type_enc  
2997 -0.034638  0.965010  
4871  0.

In [11]:
print("Original Test Data:")
print(X_test.head())

print("\nNoisy Test Data:")
print(X_noise.head())

Original Test Data:
      Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
2997                300.5                    309.8                    1345   
4871                303.7                    312.4                    1513   
3858                302.5                    311.4                    1559   
951                 295.6                    306.3                    1509   
6463                300.5                    310.0                    1358   

      Torque [Nm]  Tool wear [min]  TWF  HDF  PWF  OSF  RNF  Type_enc  
2997         62.7              153    0    0    0    0    0         1  
4871         40.1              135    0    0    0    0    0         1  
3858         37.6              209    0    0    0    0    0         1  
951          35.8               60    0    0    0    0    0         0  
6463         60.4              102    0    0    0    0    0         0  

Noisy Test Data:
      Air temperature [K]  Process temperature [K]  Rotationa

In [12]:
numeric_cols = X_test.select_dtypes(include=[np.number]).columns

difference = X_noise[numeric_cols] - X_test[numeric_cols]

print(difference.head())

      Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
2997            -0.061225                 0.055292                0.015769   
4871             0.092539                -0.026359                0.058785   
3858            -0.007118                 0.018219               -0.050183   
951             -0.008205                 0.021683                0.038779   
6463            -0.021910                 0.003828                0.043887   

      Torque [Nm]  Tool wear [min]       TWF       HDF       PWF       OSF  \
2997    -0.048971         0.042941  0.078106  0.015775  0.102924  0.042341   
4871     0.026329         0.027651 -0.004178 -0.027689  0.061874  0.083592   
3858    -0.046025        -0.014099 -0.007904  0.104668  0.005857 -0.025344   
951     -0.007871         0.019685 -0.002692 -0.052793 -0.042002  0.007561   
6463    -0.067240         0.024204  0.061908  0.000424 -0.034622  0.057964   

           RNF  Type_enc  
2997 -0.034638 -0.034990  
4871  0.

## Gaussian Noise Injection

This experiment evaluates the robustness of the trained LightGBM model by adding Gaussian noise to the numerical features of the test dataset.

- Mean (μ) = 0
- Standard Deviation (σ) = noise_level

The noisy dataset simulates sensor measurement errors commonly observed in real-world IoT systems. Comparing the original and noisy datasets helps assess how stable the model remains under different noise conditions before evaluating multiple noise levels in the next steps.

In [13]:
print("Original Test Shape:", X_test.shape)
print("Noisy Test Shape   :", X_noise.shape)

print("\nMissing values in noisy data:")
print(X_noise.isnull().sum().sum())

print("\nData types:")
print(X_noise.dtypes)

Original Test Shape: (2000, 11)
Noisy Test Shape   : (2000, 11)

Missing values in noisy data:
0

Data types:
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]     float64
Torque [Nm]                float64
Tool wear [min]            float64
TWF                        float64
HDF                        float64
PWF                        float64
OSF                        float64
RNF                        float64
Type_enc                   float64
dtype: object


# Week 4 Day 2 - Noise Level Evaluation

This section evaluates the trained model under three Gaussian noise levels:

- Low Noise: 0.05
- Medium Noise: 0.15
- High Noise: 0.30

The objective is to measure Macro F1 degradation after adding noise to the test dataset.

In [14]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [16]:
def inject_noise(X_test, noise_level):
    X_noisy = X_test.copy()

    numeric_cols = X_noisy.select_dtypes(include=[np.number]).columns
    X_noisy[numeric_cols] = X_noisy[numeric_cols].astype(float)

    noise = np.random.normal(
        loc=0,
        scale=noise_level,
        size=X_noisy[numeric_cols].shape
    )

    X_noisy.loc[:, numeric_cols] += noise

    return X_noisy

In [17]:
from sklearn.metrics import f1_score

# Prediction on clean test data
y_pred_clean = model.predict(X_test)

# Macro F1 Score
clean_macro_f1 = f1_score(
    y_test,
    y_pred_clean,
    average="macro"
)

print(f"Clean Test Macro F1: {clean_macro_f1:.4f}")

Clean Test Macro F1: 0.9923


In [18]:
noise_levels = {
    "Low": 0.05,
    "Medium": 0.15,
    "High": 0.30
}

print(noise_levels)

{'Low': 0.05, 'Medium': 0.15, 'High': 0.3}


In [19]:
# Apply low Gaussian noise
X_low_noise = inject_noise(X_test, noise_levels["Low"])

# Predict using trained model
y_pred_low = model.predict(X_low_noise)

# Calculate Macro F1
low_macro_f1 = f1_score(
    y_test,
    y_pred_low,
    average="macro"
)

print(f"Low Noise Macro F1: {low_macro_f1:.4f}")

Low Noise Macro F1: 0.9923
